# EMG Processing — BrainAccess Recording: Calf (Gastrocnemius / Soleus)

This notebook applies **the same processing pipeline** as `01_kaggle_hand_gestures_processing.ipynb` to EMG recorded with a **BrainAccess** device (Neurotechnology). All steps reuse the shared functions in `emg_methods.py`, so the results of the two datasets can be compared directly.

### Muscle
The calf (*triceps surae*) consists of:

* **Gastrocnemius** (medial and lateral heads) — superficial, crosses both the knee and the ankle. The main plantar flexor during fast, dynamic movements (heel raise, push-off in walking and running).
* **Soleus** — lies deeper, under the gastrocnemius, and crosses only the ankle. Mostly slow fibres, active in posture and standing.

The calf is a large muscle with a thick layer of subcutaneous tissue, so amplitudes are often lower and more susceptible to motion artefacts when the foot hits the ground.

### Electrode placement (SENIAM-based recommendation)
* **Gastrocnemius medialis:** on the most prominent bulge of the medial head (SENIAM).
* **Soleus:** at two thirds of the line from the medial condyle of the femur to the medial malleolus.
* **Reference:** the tibia (shin bone) or the malleolus.

### Suggested protocol
standing rest → heel raise (plantar flexion) → hold 2–3 s → lower, repeated 10×; optionally a short walking segment

**What to expect:** A burst for each heel raise. Gastrocnemius usually shows larger, more phasic bursts, while soleus activity is more tonic.

### Where to put the data
Copy the recordings into **`data/brainaccess/calf/`**. Supported formats (see `emg_io.load_brainaccess_recording`):

* `*-raw.fif` — written by `brainaccess.utils.acquisition.EEG().data.save(...)` and BrainAccess Board (MNE format, volts, annotations included)
* `.edf` / `.bdf`
* `.csv` — one column per channel in µV, plus an optional `time` column (s or ms) and an optional `annotation`/`marker` column

Auxiliary BrainAccess channels (sample counter, accelerometer, streaming flag) are dropped automatically.

### Pipeline
1. Load & inspect the recording · 2. Signal-quality check · 3. Preprocessing · 4. Envelopes · 5. Activation (burst) detection · 6. Features per burst · 7. Windowed features · 8. Fatigue analysis · 9. Rest vs. contraction classification · 10. Comparison with the Kaggle dataset

## 0. Configuration
Change these values to select a different file, channels or mains frequency.

In [ ]:
MUSCLE = 'calf'
RECORDING_FILE = None        # None -> newest file in data/brainaccess/calf/, or e.g. '20261013_1000-raw.fif'
CHANNELS = None              # None -> all EMG channels; or a list of channel names, e.g. ['ch_0', 'ch_1']
MAIN_CHANNEL = None          # channel used for the single-channel plots; None -> the most active channel
MAINS = 50.0                 # power-line frequency: 50 Hz in Europe, 60 Hz in the Americas
CSV_FS = None                # sampling rate for CSV files without a time column (BrainAccess default: 250 Hz)
LABEL_SOURCE = 'activity'    # 'activity' (detected rest/contraction) or 'annotations' (markers set during recording)
ACTIVATION_K = 3.0           # onset threshold = baseline mean + K * SD of the envelope

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'emg_methods.py').exists())
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import KFold, cross_val_predict

import emg_methods as em
import emg_io as eio
import emg_plotting as ep

ep.set_style()
# EMG_BRAINACCESS_DIR can point the notebook to another data root (e.g. for testing).
DATA_ROOT = Path(os.environ.get('EMG_BRAINACCESS_DIR', REPO_ROOT / 'data' / 'brainaccess'))
DATA_DIR = DATA_ROOT / MUSCLE
RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

## 1. Load the recording

In [ ]:
files = eio.find_recordings(DATA_DIR) if DATA_DIR.exists() else []
if not files:
    raise FileNotFoundError(
        f'No BrainAccess recording found in {DATA_DIR}.\n'
        f'Copy a *-raw.fif, .edf or .csv file there and re-run the notebook.'
    )
print('Available recordings:')
for f in files:
    print('  ', f.relative_to(DATA_DIR))

path = DATA_DIR / RECORDING_FILE if RECORDING_FILE else files[0]
rec = eio.load_brainaccess_recording(path, channels=CHANNELS, fs=CSV_FS)
FS = rec.fs
t = rec.times
emg_raw = rec.data                    # (n_channels, n_samples) in µV

print(f'\nFile:        {path.name}')
print(f'Channels:    {rec.ch_names}')
print(f'Sampling:    {FS:.1f} Hz (Nyquist {FS / 2:.1f} Hz)')
print(f'Duration:    {rec.duration_s:.1f} s ({emg_raw.shape[1]} samples)')
print(f'Annotations: {len(rec.annotations)}')
rec.annotations.head(10)

In [ ]:
pd.DataFrame({
    'mean (µV)': emg_raw.mean(axis=1),
    'std (µV)': emg_raw.std(axis=1),
    'min (µV)': emg_raw.min(axis=1),
    'max (µV)': emg_raw.max(axis=1),
    'NaN samples': np.isnan(emg_raw).sum(axis=1),
}, index=rec.ch_names).round(2)

In [ ]:
fig, axes = ep.plot_channels(t, emg_raw, rec.ch_names, f'Raw EMG — {MUSCLE} ({path.name})')
for ax in axes:
    ep.plot_annotations(ax, rec.annotations)
plt.show()

## 2. Signal-quality check

The Kaggle notebook showed that the **real sampling rate** must be checked before filtering. For BrainAccess the rate is stored in the file (typically **250 Hz** → Nyquist 125 Hz), so the textbook 20–450 Hz EMG band again cannot be used, and the upper cut-off is limited to ~112 Hz.

We also check, per channel:
* **DC offset and drift**, typical for dry/EEG-type electrodes
* **Mains interference** — fraction of power at 50 Hz (and 100 Hz)
* **Clipping** — samples stuck at the extreme values
* **Flat channels** — disconnected electrodes

In [ ]:
BAND = em.default_emg_band(FS)
f_q, p_q = em.compute_psd(em.remove_dc_offset(emg_raw), FS, nperseg=int(2 * FS))

def band_power(f, p, lo, hi):
    return p[..., (f >= lo) & (f < hi)].sum(axis=-1)

total = band_power(f_q, p_q, 1, FS / 2)
mains_power = sum(band_power(f_q, p_q, h - 1, h + 1) for h in np.arange(MAINS, FS / 2, MAINS))
quality = pd.DataFrame({
    'DC offset (µV)': emg_raw.mean(axis=1),
    'drift < 10 Hz (%)': 100 * band_power(f_q, p_q, 0.5, 10) / total,
    'mains (%)': 100 * mains_power / total,
    'EMG band (%)': 100 * band_power(f_q, p_q, *BAND) / total,
    'clipped (%)': 100 * ((emg_raw >= emg_raw.max(axis=1, keepdims=True)) | (emg_raw <= emg_raw.min(axis=1, keepdims=True))).mean(axis=1),
    'flat': emg_raw.std(axis=1) < 1e-3,
}, index=rec.ch_names)
print(f'Usable EMG band at {FS:.0f} Hz: {BAND[0]:.0f}–{BAND[1]:.1f} Hz')
quality.round(2)

**How to read the table.** A large DC offset and drift are normal before filtering. A *mains (%)* value above ~20 % means poor skin contact or long cables: check the electrode impedance in BrainAccess Board. *Clipped* should be close to 0 %. A *flat* channel is disconnected.

In [ ]:
# Pick the channel shown in the single-channel plots.
if MAIN_CHANNEL is None:
    filt_tmp = em.butter_bandpass_filter(em.remove_dc_offset(emg_raw), FS, *BAND)
    ch_idx = int(np.argmax(np.percentile(np.abs(filt_tmp), 99, axis=1) / np.median(np.abs(filt_tmp), axis=1)))
else:
    ch_idx = rec.ch_names.index(MAIN_CHANNEL)
ch_name = rec.ch_names[ch_idx]
x_raw = emg_raw[ch_idx]
print('Main channel:', ch_name)

zoom_start = t[np.argmax(np.abs(x_raw - np.median(x_raw)))] - 2
zoom = (t >= max(0, zoom_start)) & (t < max(0, zoom_start) + 4)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2), gridspec_kw={'width_ratios': [2.2, 1]})
axes[0].plot(t[zoom], x_raw[zoom], color=ep.PALETTE[0], lw=0.7)
axes[0].set_title(f'{ch_name} — 4 s zoom around the largest burst', loc='left')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('µV')
axes[1].hist(x_raw - x_raw.mean(), bins=80, color=ep.PALETTE[0])
axes[1].set_title('Amplitude histogram (DC removed)', loc='left'); axes[1].set_xlabel('µV')
fig.tight_layout(); plt.show()

## 3. Preprocessing

Same steps as in the Kaggle notebook, with the band adapted to the BrainAccess sampling rate:

| Step | Setting |
|---|---|
| DC offset removal | per channel |
| Band-pass | 4th-order Butterworth, zero-phase, **20 Hz – 0.9 × Nyquist** |
| Mains notch | 50 Hz **and its harmonics below Nyquist** (100 Hz at fs = 250 Hz) |
| Rectification | full-wave |

In [ ]:
stages = em.preprocess_emg(emg_raw, FS, band=BAND, mains=MAINS)
emg_filt = stages['filtered']
emg_rect = stages['rectified']
print(f'Band-pass {BAND[0]:.0f}–{BAND[1]:.1f} Hz, notch at', list(np.arange(MAINS, 0.98 * FS / 2, MAINS)), 'Hz')

names = ['Raw', 'DC removed', f'Band-pass {BAND[0]:.0f}–{BAND[1]:.0f} Hz', f'Notch {MAINS:.0f} Hz + harmonics', 'Rectified']
series = [emg_raw, stages['dc_removed'], stages['bandpassed'], emg_filt, emg_rect]
fig, axes = plt.subplots(len(series), 1, figsize=(12, 9), sharex=True)
for ax, name, s in zip(axes, names, series):
    ax.plot(t, s[ch_idx], color=ep.PALETTE[0], lw=0.5)
    ax.set_ylabel(f'{name}\nµV', rotation=0, ha='right', va='center')
axes[0].set_title(f'Preprocessing stages — {ch_name}', loc='left')
axes[-1].set_xlabel('Time (s)')
fig.tight_layout(); plt.show()

In [ ]:
f_raw, p_raw = em.compute_psd(stages['dc_removed'], FS, nperseg=512)
f_flt, p_flt = em.compute_psd(emg_filt, FS, nperseg=512)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.semilogy(f_raw, p_raw[ch_idx], color=ep.INK_MUTED, lw=1.2, label='DC removed (unfiltered)')
ax.semilogy(f_flt, p_flt[ch_idx], color=ep.PALETTE[0], lw=2, label='Band-pass + notch')
ax.axvspan(*BAND, color=ep.PALETTE[0], alpha=0.06, lw=0)
for h in np.arange(MAINS, FS / 2, MAINS):
    ax.axvline(h, color=ep.INK_MUTED, ls='--', lw=0.8)
    ax.text(h, 1, f' {h:.0f} Hz', transform=ax.get_xaxis_transform(), va='top', fontsize=8, color=ep.INK_SECONDARY)
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('PSD (µV²/Hz)')
ax.set_title(f'Welch PSD — {ch_name}', loc='left')
ax.legend(); plt.show()

In [ ]:
# Filtered signal of all channels.
fig, axes = ep.plot_channels(t, emg_filt, rec.ch_names, f'Filtered EMG — {MUSCLE}')
for ax in axes:
    ep.plot_annotations(ax, rec.annotations)
plt.show()

## 4. Envelope detection

In [ ]:
WIN_ENV = int(0.15 * FS)   # 150 ms
x_f, x_r = emg_filt[ch_idx], emg_rect[ch_idx]
env = {
    'Moving average (150 ms)': em.moving_average_envelope(x_r, WIN_ENV),
    'RMS (150 ms)': em.rms_envelope(x_f, WIN_ENV),
    'Hilbert': em.hilbert_envelope(x_f),
    'Linear envelope (5 Hz LP)': em.lowpass_envelope(x_f, FS, cutoff=5),
}

fig, axes = plt.subplots(len(env), 1, figsize=(12, 7), sharex=True, sharey=True)
for ax, (name, e), c in zip(axes, env.items(), ep.PALETTE):
    ax.plot(t, x_r, color=ep.GRID, lw=0.5)
    ax.plot(t, e, color=c, lw=1.4)
    ax.set_title(name, loc='left', fontsize=9)
    ax.set_ylabel('µV')
axes[0].set_ylim(0, np.percentile(x_r, 99.8))
axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'Envelope comparison on the rectified signal (grey) — {ch_name}', x=0.01, ha='left', fontsize=11, fontweight='bold')
fig.tight_layout(); plt.show()

## 5. Activation (burst) detection

The Kaggle data came with gesture labels. Our recordings have, at most, manual markers. We therefore detect muscle activity **automatically** from the linear envelope:

* baseline = the quietest 10 % of the envelope (resting muscle)
* threshold = baseline mean + K · SD (K = `ACTIVATION_K`)
* bursts shorter than 200 ms are dropped, and gaps shorter than 200 ms are merged

This is the classic threshold method (Hodges & Bui, 1996).

In [ ]:
envelope = env['Linear envelope (5 Hz LP)']
bursts = em.detect_activations(envelope, FS, k=ACTIVATION_K)
active = em.activity_mask(len(envelope), bursts)
print(f"Threshold: {bursts.attrs['threshold']:.2f} µV — {len(bursts)} bursts detected, "
      f"{100 * active.mean():.0f} % of the recording is active")

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(t, x_r, color=ep.GRID, lw=0.5)
ax.plot(t, envelope, color=ep.PALETTE[0], lw=1.4, label='Linear envelope')
ax.axhline(bursts.attrs['threshold'], color=ep.PALETTE[1], lw=1, ls='--', label='Threshold')
for i, b in enumerate(bursts.itertuples(), start=1):
    ax.axvspan(b.onset_s, b.offset_s, color=ep.PALETTE[0], alpha=0.08, lw=0)
    ax.text((b.onset_s + b.offset_s) / 2, 1.0, str(i), transform=ax.get_xaxis_transform(), ha='center', va='top', fontsize=8, color=ep.INK_SECONDARY)
ep.plot_annotations(ax, rec.annotations, labels=False)
ax.set_ylim(0, np.percentile(x_r, 99.8))
ax.set_title(f'Detected bursts — {ch_name}', loc='left'); ax.set_xlabel('Time (s)'); ax.set_ylabel('µV')
ax.legend(loc='upper right'); plt.show()

Dashed lines are the annotation markers. If the detection misses weak contractions, lower `ACTIVATION_K` (e.g. 2). If noise is detected as activity, raise it (e.g. 5). Comparing the detected bursts with the markers is a quick way to validate the detector.

## 6. Features per burst and at rest

The same time- and frequency-domain descriptors as in the Kaggle notebook are computed on the **filtered (non-rectified)** signal of each burst, and of the resting periods (> 0.5 s away from any burst).

In [ ]:
WAMP_THR = 0.1 * x_f.std()
rest_mask = ~em.activity_mask(len(x_f), bursts.assign(onset=(bursts.onset - int(0.5 * FS)).clip(lower=0),
                                                       offset=bursts.offset + int(0.5 * FS)))
segments = {'rest': x_f[rest_mask]}
for i, b in enumerate(bursts.itertuples(), start=1):
    segments[f'burst {i}'] = x_f[int(b.onset):int(b.offset)]

rows = []
for name, seg in segments.items():
    if seg.size < 64:
        continue
    feats = em.compute_time_domain_features(seg, wamp_threshold=WAMP_THR)
    feats.update(em.compute_frequency_domain_features(seg, FS, nperseg=min(256, seg.size)))
    dur = seg.size / FS
    for k in ['IEMG', 'Signal Energy', 'WL', 'ZC', 'SSC', 'WAMP']:
        feats[k] /= dur
    rows.append({'segment': name, 'duration_s': dur, **feats})

burst_features = pd.DataFrame(rows).set_index('segment')
burst_features.to_csv(RESULTS_DIR / f'brainaccess_{MUSCLE}_burst_features.csv')
burst_features.rename(columns={k: f'{k} /s' for k in ['IEMG', 'Signal Energy', 'WL', 'ZC', 'SSC', 'WAMP']}).round(2)

In [ ]:
only_bursts = burst_features.drop(index='rest', errors='ignore')
snr_db = 20 * np.log10(only_bursts['RMS'] / burst_features.loc['rest', 'RMS']) if 'rest' in burst_features.index else None

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharex=True)
for ax, col, unit in zip(axes, ['RMS', 'MDF', 'MNF'], ['µV', 'Hz', 'Hz']):
    ax.plot(range(1, len(only_bursts) + 1), only_bursts[col], 'o-', color=ep.PALETTE[0], ms=6, lw=1.5)
    if 'rest' in burst_features.index:
        ax.axhline(burst_features.loc['rest', col], color=ep.INK_MUTED, ls='--', lw=1)
        ax.text(1, burst_features.loc['rest', col], ' rest', va='bottom', fontsize=8, color=ep.INK_SECONDARY)
    ax.set_title(f'{col} ({unit})', loc='left'); ax.set_xlabel('Burst #')
fig.suptitle(f'Feature evolution over consecutive bursts — {ch_name}', x=0.01, ha='left', fontsize=11, fontweight='bold')
fig.tight_layout(); plt.show()
if snr_db is not None:
    print(f'Burst-to-rest RMS ratio (SNR): median {snr_db.median():.1f} dB, range {snr_db.min():.1f}–{snr_db.max():.1f} dB')

* RMS shows how strong each contraction was. Repeated contractions of equal effort should give similar values.
* A **downward trend of MDF/MNF across bursts** suggests accumulating fatigue (Section 8).
* The burst-to-rest RMS ratio (SNR) measures recording quality. Values above ~20 dB are good for surface EMG.

## 7. Windowed feature extraction

Identical to the Kaggle notebook: **250 ms windows, 125 ms step**, and MAV, RMS, VAR, WL, ZC, SSC, WAMP, MNF and MDF per channel. Each window is labelled according to `LABEL_SOURCE`:

* `'activity'` — *rest* / *contraction* from the burst detector (windows that only partly overlap a burst are dropped)
* `'annotations'` — the most recent marker before the window (e.g. markers such as `fist`, `flexion` set with `eeg.annotate(...)` during the recording)

In [ ]:
WIN, STEP = int(0.25 * FS), int(0.125 * FS)
features = em.windowed_features(emg_filt, FS, WIN, STEP, rec.ch_names)
starts = features['start'].to_numpy()

if LABEL_SOURCE == 'annotations' and len(rec.annotations):
    ann = rec.annotations.sort_values('onset_s')
    idx = np.searchsorted(ann['onset_s'].to_numpy(), starts / FS, side='right') - 1
    features['label'] = np.where(idx >= 0, ann['label'].to_numpy()[np.clip(idx, 0, None)], 'before first marker')
else:
    frac_active = np.array([active[s:s + WIN].mean() for s in starts])
    features['label'] = np.select([frac_active == 0, frac_active == 1], ['rest', 'contraction'], default='transition')
features = features[features['label'] != 'transition'].reset_index(drop=True)

FEATURE_COLS = [c for c in features.columns if c not in ('start', 't_center', 'label')]
features.to_csv(RESULTS_DIR / f'brainaccess_{MUSCLE}_window_features.csv', index=False)
print(f'{len(features)} windows × {len(FEATURE_COLS)} features')
features['label'].value_counts()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, feat in zip(axes, ['RMS', 'WL', 'MDF']):
    col = f'{ch_name}_{feat}'
    groups = sorted(features['label'].unique())
    ax.boxplot([features.loc[features['label'] == g, col] for g in groups], tick_labels=groups, widths=0.5,
               medianprops={'color': ep.PALETTE[1], 'lw': 2}, flierprops={'ms': 2, 'markeredgecolor': ep.INK_MUTED})
    ax.set_title(f'{feat} per window', loc='left'); ax.grid(axis='x', visible=False)
fig.suptitle(f'Window features by class — {ch_name}', x=0.01, ha='left', fontsize=11, fontweight='bold')
fig.tight_layout(); plt.show()

## 8. Fatigue analysis

During a sustained contraction the conduction velocity of muscle fibres decreases, and the EMG spectrum shifts towards **lower frequencies**. MDF and MNF fall, while RMS often rises as more motor units are recruited. We track both within the **longest burst** using 1 s windows with 0.5 s step, and fit a linear trend.

In [ ]:
longest = bursts.loc[bursts['duration_s'].idxmax()] if len(bursts) else None
if longest is None or longest.duration_s < 3:
    print('No contraction longer than 3 s — record a sustained isometric hold to analyse fatigue.')
else:
    seg = x_f[int(longest.onset):int(longest.offset)]
    track = em.spectral_fatigue_track(seg, FS, window_s=1.0, step_s=0.5, nperseg=int(FS / 2))
    track['t_s'] += longest.onset_s

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), sharex=True)
    for ax, col, unit, c in zip(axes, ['MDF', 'RMS'], ['Hz', 'µV'], [ep.PALETTE[0], ep.PALETTE[1]]):
        fit = stats.linregress(track['t_s'], track[col])
        ax.plot(track['t_s'], track[col], 'o', color=c, ms=5)
        ax.plot(track['t_s'], fit.intercept + fit.slope * track['t_s'], color=ep.INK_SECONDARY, lw=1)
        ax.set_title(f'{col} ({unit}): slope {fit.slope:+.2f} {unit}/s, p = {fit.pvalue:.3g}', loc='left')
        ax.set_xlabel('Time (s)')
    fig.suptitle(f'Fatigue within the longest burst ({longest.duration_s:.1f} s) — {ch_name}', x=0.01, ha='left', fontsize=11, fontweight='bold')
    fig.tight_layout(); plt.show()

    mdf_fit = stats.linregress(track['t_s'], track['MDF'])
    drop = 100 * mdf_fit.slope * (track['t_s'].iloc[-1] - track['t_s'].iloc[0]) / track['MDF'].iloc[:3].mean()
    print(f'MDF change over the contraction: {drop:+.1f} % '
          f"({'consistent with fatigue' if mdf_fit.slope < 0 and mdf_fit.pvalue < 0.05 else 'no significant fatigue trend'})")

## 9. Rest vs. contraction classification

The same Random Forest as in the Kaggle notebook. With a single recording there is no second session or subject to test on, so we use **5-fold cross-validation on contiguous time blocks** (no shuffling). Shuffling would put neighbouring, overlapping windows into both train and test, and inflate accuracy.

In [ ]:
y = features['label'].to_numpy()
classes = sorted(np.unique(y))
if len(classes) < 2 or pd.Series(y).value_counts().min() < 5:
    print('Not enough windows per class for classification:', pd.Series(y).value_counts().to_dict())
else:
    clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
    pred = cross_val_predict(clf, features[FEATURE_COLS], y, cv=KFold(n_splits=5, shuffle=False))
    print(f'Accuracy {accuracy_score(y, pred):.3f}, balanced accuracy {balanced_accuracy_score(y, pred):.3f} '
          f'(chance {1 / len(classes):.3f})')
    ep.plot_confusion(confusion_matrix(y, pred, labels=classes), classes, 'Confusion matrix — contiguous 5-fold CV')
    plt.show()

    clf.fit(features[FEATURE_COLS], y)
    imp = pd.Series(clf.feature_importances_, index=FEATURE_COLS).groupby(lambda c: c.rsplit('_', 1)[1]).sum().sort_values()
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(imp.index, imp.to_numpy(), color=ep.PALETTE[0], height=0.6)
    ax.set_title('Importance by feature type', loc='left'); ax.grid(axis='y', visible=False)
    fig.tight_layout(); plt.show()

## 10. Comparison with the Kaggle (Myo) dataset

Absolute amplitudes are not comparable between devices: electrodes, gain and skin preparation all differ. **Relative** measures are comparable: the contraction-to-rest RMS ratio (SNR) and the spectral content of the active signal. The Kaggle numbers are read from `results/kaggle_gestures_window_features.csv`, which notebook 01 writes.

In [ ]:
def summarise(df, channels, active_mask, name, fs, band):
    rms = df[[f'{c}_RMS' for c in channels]].mean(axis=1)
    mdf = df[[f'{c}_MDF' for c in channels]].mean(axis=1)
    return {
        'dataset': name,
        'fs (Hz)': fs,
        'band (Hz)': f'{band[0]:.0f}–{band[1]:.0f}',
        'rest RMS (µV)': rms[~active_mask].median(),
        'active RMS (µV)': rms[active_mask].median(),
        'SNR (dB)': 20 * np.log10(rms[active_mask].median() / rms[~active_mask].median()),
        'active MDF (Hz)': mdf[active_mask].median(),
    }

rows = []
if set(features['label']) >= {'rest', 'contraction'}:
    rows.append(summarise(features, rec.ch_names, (features['label'] == 'contraction').to_numpy(),
                          f'BrainAccess — {MUSCLE}', FS, BAND))
kaggle_csv = RESULTS_DIR / 'kaggle_gestures_window_features.csv'
if kaggle_csv.exists():
    kg = pd.read_csv(kaggle_csv)
    rows.append(summarise(kg, eio.MYO_CHANNELS, (kg['gesture'] != 1).to_numpy(), 'Kaggle Myo — forearm', eio.MYO_FS,
                          em.default_emg_band(eio.MYO_FS)))
else:
    print('Run notebook 01 first to include the Kaggle reference.')
pd.DataFrame(rows).set_index('dataset').round(2)

## 11. Conclusions — checklist for interpreting your recording

* **Quality (Section 2):** Is mains interference low and is there no clipping or flat channel? If not, improve the electrode contact and re-record.
* **Preprocessing (Section 3):** Does the PSD after filtering show a smooth EMG spectrum, with notches at 50/100 Hz and no remaining peaks?
* **Bursts (Section 5):** Does every contraction of the protocol appear as one detected burst, matching the annotation markers?
* **Features (Section 6):** Are RMS values similar for contractions of equal effort? Is the SNR above ~20 dB?
* **Fatigue (Section 8):** Does MDF decrease significantly during sustained contractions?
* **Comparison (Section 10):** How do SNR and MDF compare with the Myo armband? A higher sampling rate (250 vs 200 Hz) gives a wider usable band and therefore somewhat higher MDF values.

### References
* Hermens H.J. et al. (2000). *Development of recommendations for SEMG sensors and sensor placement procedures (SENIAM).* J Electromyogr Kinesiol 10(5).
* Hodges P.W., Bui B.H. (1996). *A comparison of computer-based methods for the determination of onset of muscle contraction using electromyography.* Electroencephalogr Clin Neurophysiol 101(6).
* De Luca C.J. (1997). *The use of surface electromyography in biomechanics.* J Appl Biomech 13(2).
* Cifrek M. et al. (2009). *Surface EMG based muscle fatigue evaluation in biomechanics.* Clin Biomech 24(4).